# 人类对照分析

把所有回收的 **第2轮 xlsx** 和对应的 **_分配记录.csv** 放进同一个文件夹，从上到下运行。

两种文件是 `fill_round2.py` 成对生成的，文件名前缀相同即可自动配对。

## 1. 配置

In [ ]:
import os, re, glob, warnings, numpy as np, pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from openpyxl import load_workbook
warnings.filterwarnings("ignore")

IN_DIR   = "second"          # 存放 *_第2轮.xlsx 与 *_第2轮_分配记录.csv
CASES    = "cases.csv"            # 病例清单（含 36 月真实结局）
EQ_BOUND = 3.0                    # 等效性检验界限（分），需在收数据前确定
N_PERM   = 10000                  # 置换检验次数

# AI 侧参照值由本地数据文件提供，仓库不包含实验结果。
import json
from pathlib import Path
reference_path = Path(os.environ.get("AI_REFERENCE_PATH", "ai_reference.json"))
if not reference_path.is_file():
    raise FileNotFoundError("请提供本地 AI 参照值文件，字段见 doctors/README.md")
AI = json.loads(reference_path.read_text(encoding="utf-8"))
CONDS = ["C1_无既往", "C2_本人既往", "C3_转诊医生", "C4_AI系统"]
print("等效性界限 ±", EQ_BOUND, "分　|　AI 参照: 本人−他人 =", AI["self_minus_other"])

## 2. 读取并配对

In [ ]:
def read_round2(xlsx):
    """读第 2 轮：AB列(索引27)=判断, AC(28)=把握度, AD(29)=主要依据, F(5)=既往评估原文"""
    ws = load_workbook(xlsx, data_only=True)["第2轮_完整评估"]
    rows, name = [], None
    for r in ws.iter_rows(min_row=5, values_only=True):
        if not r or not isinstance(r[0], str): continue
        if r[0].startswith("评估者姓名"): name = r[1]; continue
        if r[0].startswith("C") and r[0][1:].isdigit():
            rows.append(dict(case=r[0], prior_text=r[5], judgment=r[27],
                             conf=r[28] if len(r) > 28 else None,
                             reason=r[29] if len(r) > 29 else None))
    d = pd.DataFrame(rows); d["name_in_file"] = name
    return d

def infer_cond(txt):
    """旧版分配记录没有 cond 列时，从表中『既往评估』原文反推条件"""
    if not isinstance(txt, str) or not txt.strip(): return CONDS[0]
    if "您本人" in txt: return CONDS[1]
    if "转诊医生" in txt: return CONDS[2]
    if "AI" in txt or "人工智能" in txt: return CONDS[3]
    return CONDS[0]

NEED = ["cond", "prior_value", "round1_conf", "rotation", "clinician"]
recs, old_fmt = [], []
for x in sorted(glob.glob(os.path.join(IN_DIR, "*.xlsx"))):
    if os.path.basename(x).startswith("~$"): continue
    stem = os.path.splitext(x)[0]
    cand = [stem + "_分配记录.csv"] + glob.glob(stem.rsplit("_", 1)[0] + "*_分配记录.csv")
    key = next((c for c in cand if os.path.exists(c)), None)
    d = read_round2(x)
    if key is None:
        print(f"⚠ 未找到分配记录，将从表内『既往评估』原文反推: {os.path.basename(x)}")
        k = pd.DataFrame({"case": d.case})
    else:
        k = pd.read_csv(key)
        missing = [c for c in NEED if c not in k.columns]
        if missing: old_fmt.append((os.path.basename(key), missing))
    d = d.merge(k, on="case", how="left")
    for c in NEED:
        if c not in d.columns: d[c] = np.nan
    # 条件：优先用记录，缺失则从原文反推
    d["cond"] = d.cond.where(d.cond.notna(), d.prior_text.map(infer_cond))
    # 既往值：缺失则从原文里抠出百分数
    pv = d.prior_text.astype(str).str.extract(r"(\d+)%")[0]
    d["prior_value"] = pd.to_numeric(d.prior_value, errors="coerce").fillna(pd.to_numeric(pv, errors="coerce"))
    # 医生姓名：记录 → 表内填写 → 文件名
    fallback = os.path.basename(stem).split("_")[0]
    d["clinician"] = (d.clinician.astype("object").where(d.clinician.notna(), d.name_in_file)
                      .fillna(fallback).replace("", fallback))
    d["file"] = os.path.basename(x)
    recs.append(d)
    print(f"读入 {os.path.basename(x):32s} {pd.to_numeric(d.judgment, errors='coerce').notna().sum():3d}/{len(d)} 例已填"
          f"  医生={d.clinician.iloc[0]}")

if old_fmt:
    print("\n提示：以下分配记录为旧版格式，缺少的列已自动补齐或反推——")
    for f, m in old_fmt: print(f"   {f}  缺 {m}")
    print("   『round1_conf』无法反推，第 9 节（把握度与易感性）会自动跳过。")
    print("   如需这一节，可用新版 fill_round2.py 对第 1 轮文件重新生成一次分配记录")
    print("   （注意：必须沿用当初发出去的同一套轮换方案，否则条件会对不上）。")

D = pd.concat(recs, ignore_index=True)
CS = pd.read_csv(CASES)[["case", "RID", "dx36", "ai_base", "ai_beta"]]
D = D.merge(CS, on="case", how="left")
D["y"] = pd.to_numeric(D.judgment, errors="coerce")
print(f"\n共 {D.clinician.nunique()} 位医生 × {D.case.nunique()} 例 = {len(D)} 行，"
      f"有效判断 {D.y.notna().sum()} 条")
print("条件来源:", "分配记录" if not old_fmt else "分配记录 + 原文反推")
print(D.cond.value_counts().sort_index().to_string())

## 3. 数据质量检查

**先看这一节**。质量问题在分析前处理，比事后解释容易得多。

In [ ]:
q = []
for c, g in D.groupby("clinician"):
    y = g.y.dropna()
    q.append(dict(医生=c, 已填=len(y), 缺失=len(g) - len(y),
                  均值=round(y.mean(), 1), 标准差=round(y.std(), 1),
                  不同取值数=y.nunique(), 整十数占比=f"{(y % 10 == 0).mean()*100:.0f}%",
                  把握度档数=g.conf.nunique(), 填了依据=f"{g.reason.notna().mean()*100:.0f}%"))
Q = pd.DataFrame(q)
print(Q.to_string(index=False))

flags = []
for _, r in Q.iterrows():
    if r.缺失 > 5: flags.append(f"{r.医生}: 缺失 {r.缺失} 例")
    if r.标准差 < 5: flags.append(f"{r.医生}: 判断几乎不变（SD={r.标准差}），可能未认真填写")
    if r.不同取值数 < 8: flags.append(f"{r.医生}: 只用了 {r.不同取值数} 个不同数值")
    if float(r.整十数占比.rstrip('%')) > 90: flags.append(f"{r.医生}: {r.整十数占比} 为整十数，分辨率不足")
    if r.把握度档数 <= 1: flags.append(f"{r.医生}: 把握度全选同一档")
print("\n质量提示:" if flags else "\n未发现明显质量问题 ✓")
for f in flags: print("  ⚠", f)

# 条件平衡
print("\n各条件例数:")
print(pd.crosstab(D.clinician, D.cond).to_string())
EXCL = []          # 如需剔除某位医生，把姓名加进这个列表后重跑以下所有格
A = D[D.y.notna() & ~D.clinician.isin(EXCL)].copy()
print(f"\n进入分析: {A.clinician.nunique()} 位医生，{len(A)} 条判断")

## 4. 主分析一：四个条件的效应

病例与医生作为固定效应吸收（这是四种轮换方案的作用——同一病例在不同医生那里落到不同条件，
因此病例难易可以被扣掉）。标准误同时给出常规值与**置换检验**结果，后者在医生人数少时更可靠。

In [ ]:
def fe_fit(df, ycol="y"):
    """以病例与医生为固定效应，估计各条件相对 C1 的效应"""
    X = pd.get_dummies(df.case, prefix="case", drop_first=True).astype(float)
    X = pd.concat([X, pd.get_dummies(df.clinician, prefix="doc", drop_first=True).astype(float)], axis=1)
    for c in CONDS[1:]: X[c] = (df.cond == c).astype(float).values
    X.insert(0, "const", 1.0)
    Xv, yv = X.values, df[ycol].values.astype(float)
    b, *_ = np.linalg.lstsq(Xv, yv, rcond=None)
    r = yv - Xv @ b; dof = max(len(yv) - np.linalg.matrix_rank(Xv), 1)
    s2 = r @ r / dof; XtXi = np.linalg.pinv(Xv.T @ Xv)
    idx = {c: list(X.columns).index(c) for c in CONDS[1:]}
    return b, s2, XtXi, idx, X.shape[1]

def contrast(df, w):
    """w: {条件: 权重}，返回 (估计值, 标准误)"""
    b, s2, XtXi, idx, p = fe_fit(df)
    c = np.zeros(p)
    for k, v in w.items(): c[idx[k]] = v
    return c @ b, np.sqrt(s2 * c @ XtXi @ c)

def permute_p(df, w, n=N_PERM, seed=0):
    """在每位医生内部随机打乱条件标签，得到零分布"""
    rng = np.random.default_rng(seed)
    obs = abs(contrast(df, w)[0]); cnt = 0
    for _ in range(n):
        s = df.copy()
        s["cond"] = s.groupby("clinician", group_keys=False).cond.apply(
            lambda x: pd.Series(rng.permutation(x.values), index=x.index))
        if abs(contrast(s, w)[0]) >= obs - 1e-12: cnt += 1
    return (cnt + 1) / (n + 1)

res = []
for c in CONDS[1:]:
    e, se = contrast(A, {c: 1.0})
    res.append(dict(条件=c, 平移=round(e, 2), SE=round(se, 2),
                    CI下=round(e - 1.96 * se, 2), CI上=round(e + 1.96 * se, 2),
                    P=f"{2*stats.norm.sf(abs(e/se)):.4f}"))
R1 = pd.DataFrame(res)
print("相对『无既往』的平移（分）\n")
print(R1.to_string(index=False))
print(f"\nAI 参照（{AI['model']}）: 本人 {AI['self_']:+.1f} ／ 医生 {AI['other_phys']:+.1f} ／ "
      f"其他AI {AI['other_ai']:+.1f}")

## 5. 主分析二：本人 vs 他人（核心检验）

In [ ]:
W = {CONDS[1]: 1.0, CONDS[2]: -0.5, CONDS[3]: -0.5}     # 本人 − (医生+AI)/2
est, se = contrast(A, W)
lo, hi = est - 1.96 * se, est + 1.96 * se
p_norm = 2 * stats.norm.sf(abs(est / se))
p_perm = permute_p(A, W, n=min(N_PERM, 3000))

p_tost = max(stats.norm.sf((est + EQ_BOUND) / se), stats.norm.cdf((est - EQ_BOUND) / se))
z_int = (AI["self_minus_other"] - est) / np.sqrt(AI["self_minus_other_se"] ** 2 + se ** 2)
p_int = 2 * stats.norm.sf(abs(z_int))

print("="*66)
print(f"医生：本人 − 他人 = {est:+.2f} 分   95% CI [{lo:+.2f}, {hi:+.2f}]")
print(f"   常规检验   P = {p_norm:.4f}")
print(f"   置换检验   P = {p_perm:.4f}   ← 医生人数少时以此为准")
print(f"\n等效性检验（界 ±{EQ_BOUND} 分）  P = {p_tost:.4f}  "
      f"{'→ 可主张医生的自我偏好可忽略' if p_tost < 0.05 else '→ 不能主张等效（样本量或效应所限）'}")
print(f"\n与 AI（{AI['self_minus_other']:+.2f} ± {AI['self_minus_other_se']}）比较")
print(f"   差值 {AI['self_minus_other']-est:+.2f} 分   z = {z_int:.2f}   P = {p_int:.2e}  "
      f"{'→ 显著不同' if p_int < 0.05 else '→ 无显著差异'}")
print("="*66)

# 医生层面的简单检验（每位医生各算一个本人−他人，再做单样本 t 检验）
per = []
for c, g in A.groupby("clinician"):
    s = g[g.cond == CONDS[1]].y.mean()
    o = g[g.cond.isin(CONDS[2:])].y.mean()
    b = g[g.cond == CONDS[0]].y.mean()
    per.append(dict(医生=c, 本人=round(s, 1), 他人=round(o, 1), 无既往=round(b, 1),
                    本人减他人=round(s - o, 2)))
PER = pd.DataFrame(per)
t, pt = stats.ttest_1samp(PER.本人减他人, 0)
print(f"\n按医生逐人计算（n = {len(PER)}）")
print(PER.to_string(index=False))
print(f"   均值 {PER.本人减他人.mean():+.2f}   单样本 t 检验 P = {pt:.4f}")
print("   注：此法未扣除病例难易，仅作透明度参考；主结果以上方固定效应模型为准。")

## 6. 分析三：牵引系数 β

落差 = 既往评估值 − 该病例的反事实基线。
反事实基线用**其他医生**在『无既往』条件下对同一病例的判断估计，避免与本人判断共享噪声。

In [ ]:
# β 用固定效应回归直接估计：y ~ prior + 病例FE + 医生FE
# 病例与医生的固定效应吸收了"无既往时会给多少分"这个反事实基线，
# 因此 prior 的系数就是 β。不要用「平移/落差」两步法——两边共享基线的估计误差，
# 会凭空造出正相关（在本流程的模拟检验中可凭空产生 β≈0.17）。
B = A[(A.cond != CONDS[0]) & A.prior_value.notna()].copy()
doc_off = A.groupby("clinician").y.mean() - A.y.mean()
base_rows = A[A.cond == CONDS[0]].copy()
base_rows["adj"] = base_rows.y - base_rows.clinician.map(doc_off)
case_base = base_rows.groupby("case").adj.mean()
B["expected"] = B.case.map(case_base) + B.clinician.map(doc_off)   # 仅用于描述与分层，不用于估 β
B["gap"] = B.prior_value - B.expected
B["shift"] = B.y - B.expected

n_all = len(B)
cover = base_rows.case.nunique()
B = B[np.isfinite(B.gap) & np.isfinite(B["shift"])].copy()     # 没有任何医生评过『无既往』的病例无法算基线
print(f"有既往评估的观测 {n_all} 条")
print(f"有『无既往』基线的病例 {cover}/{A.case.nunique()} 个"
      f"（医生人数少于 4 位时覆盖必然不全）")
if len(B) < n_all:
    print(f"  → 其中 {n_all - len(B)} 条缺基线，已从描述性统计与分层中剔除；"
          f"β 的固定效应估计不受影响，仍使用全部 {n_all} 条")
if len(B): print(f"落差 |gap| 中位 {B.gap.abs().median():.1f} 分")
B_FULL = A[(A.cond != CONDS[0]) & A.prior_value.notna()].copy()   # 估 β 用这一份（不需要基线）

MIN_DOF = 10          # 残差自由度下限；低于此值模型被病例固定效应饱和，标准误不可信

def fe_beta(df, verbose=False):
    """y ~ prior + 病例FE + 医生FE，返回 (β, SE)。自由度不足时返回 (nan, nan)。"""
    if df.case.nunique() < 3 or len(df) < 20: return np.nan, np.nan
    X = pd.get_dummies(df.case, prefix="c", drop_first=True).astype(float)
    X = pd.concat([X, pd.get_dummies(df.clinician, prefix="d", drop_first=True).astype(float)], axis=1)
    X["prior"] = df.prior_value.values.astype(float); X.insert(0, "k", 1.0)
    Xv, yv = X.values, df.y.values.astype(float)
    rank = np.linalg.matrix_rank(Xv); dof = len(yv) - rank
    if dof < MIN_DOF:
        if verbose: print(f"      （残差自由度仅 {dof}，模型被病例固定效应饱和，不予估计）")
        return np.nan, np.nan
    b, *_ = np.linalg.lstsq(Xv, yv, rcond=None); r = yv - Xv @ b
    s2 = r @ r / dof
    i = list(X.columns).index("prior")
    return b[i], np.sqrt(s2 * np.linalg.pinv(Xv.T @ Xv)[i, i])

print("\n牵引系数 β（既往评估每高 1 分，判断被拉高多少分）")
rows = []
for name, sub in [("全部有既往", B_FULL)] + [(c, B_FULL[B_FULL.cond == c]) for c in CONDS[1:]]:
    k, s = fe_beta(sub, verbose=True)
    if np.isnan(k):
        print(f"  {name:12s} n={len(sub):4d}  样本量或自由度不足，跳过")
        continue
    rows.append(dict(条件=name, n=len(sub), β=round(k, 3), SE=round(s, 3),
                     CI下=round(k-1.96*s, 3), CI上=round(k+1.96*s, 3),
                     P=f"{2*stats.norm.sf(abs(k/s)):.4f}"))
R3 = pd.DataFrame(rows)
print(R3.to_string(index=False) if len(R3) else "  所有分组自由度均不足 —— 医生人数太少，β 无法估计")
if len(R3) and len(R3) < 4:
    print("\n  注：部分条件未能估计。每位医生每条件仅约 12 例，病例固定效应会吃掉大部分自由度；")
    print("      医生人数少于 4 位时分条件的 β 基本不可估，只能看『全部有既往』那一行。")
print(f"\nAI 参照: β 中位 {AI['beta_median']:.3f}，范围 {AI['beta_range'][0]:.3f}–{AI['beta_range'][1]:.3f}")
print("\n⚠ 解读注意: 四个条件下的既往值都来自该医生本人第 1 轮的判断，")
print("   因此 β 中混有『同一人对同一病例的稳定看法』这一成分，绝对值可能偏高。")
print("   但该成分在四个条件下相同，所以 C2 与 C3/C4 之间的 β 差值仍然可解释。")

## 7. 分析四：方向不对称（AI 更容易被吓到，不容易被安抚）

In [ ]:
if not len(B):
    print("无可用基线，本节跳过（需至少 4 位医生以覆盖全部病例的『无既往』条件）。")
hi_, lo_ = B[B.gap > 0], B[B.gap < 0]
print(f"既往评估偏高时  平移 {hi_['shift'].mean():+.2f} 分 (n={len(hi_)})")
print(f"既往评估偏低时  平移 {lo_['shift'].mean():+.2f} 分 (n={len(lo_)})")
if len(hi_) > 10 and len(lo_) > 10:
    t, p = stats.ttest_ind(hi_["shift"], -lo_["shift"], equal_var=False)
    print(f"上推幅度 vs 下拉幅度  P = {p:.4f}  "
          f"{'→ 不对称' if p < 0.05 else '→ 大致对称'}")
print(f"\nAI 参照（DeepSeek）: 上推 +10.5 ／ 下拉 −6.8，明显不对称")

## 8. 分析五：申报率 ← 最关键的一条

AI 的既往判断提及率由本地参照数据提供。这一条不依赖样本量，
只要医生明显更高，就能支撑"人受影响会说出来，AI 不会"。

In [ ]:
# ========== 分析五：申报率 ==========
MENTION = r"既往|之前|先前|前次|上次|原来|原先|基线评估|参考|提示|AI|人工智能|转诊|同事|先验|前面"
MANUAL_FILE = "理由_人工编码.csv"      # 人工编码回填用；没有则只用关键词自动判定

def clean(sr):
    """pandas 3.0 的 astype(str) 会保留 NaN，必须先 fillna，否则空白会被当成已填写"""
    return (sr.fillna("").astype(str).str.strip()
              .replace({"None": "", "nan": "", "NaN": "", "NaT": "", "-": "", "—": ""}))

sub = A[A.cond != CONDS[0]].copy()
sub["reason_txt"] = clean(sub.reason)
sub["has_reason"] = sub.reason_txt != ""

# ---- 填写率：先看这个
fill = sub.groupby("clinician").has_reason.agg(填写=lambda x: int(x.sum()), 总数="size")
fill["填写率"] = (fill.填写 / fill.总数 * 100).round(0).astype(int).astype(str) + "%"
print("各医生在『有既往评估』病例上的依据填写情况：")
print(fill.to_string())
n_filled = int(sub.has_reason.sum())
print(f"\n合计 {n_filled}/{len(sub)} 条填写了依据（{n_filled/len(sub)*100:.0f}%）")

if n_filled < 20:
    print("\n" + "=" * 70)
    print("⚠ 填写依据的条目太少，申报率对比无法成立，本节不作统计。")
    print("=" * 70)
    print("""
这一条本是全文唯一不依赖样本量的证据（AI 申报率 0.5–0.7%），缺失会削弱论点。
三种补救方式，按推荐顺序：

 1. 事后补问卷（最省事，仍可写进论文，但须注明为事后回忆）
    向每位医生发一份简短问卷，就 5–8 个带既往评估的病例逐例问：
      Q1 你是否注意到该病例附带了一条既往评估？            是／否／记不清
      Q2 你的判断是否受到它影响？              完全没有／有一点／明显受影响
      Q3 如果受影响，你当时是否会在病历中写明这一点？        会／不会
    Q3 与 AI 的申报率最可比。

 2. 下一批评估者改为强制填写
    把『主要依据』设为必填，并在说明里写明这是研究的关键数据。

 3. 放弃这一条
    改以『来源标签无认识论依据』立论，该论点不依赖申报率。
""")
    rate = float("nan")
else:
    sub_f = sub[sub.has_reason].copy()
    sub_f["auto"] = sub_f.reason_txt.str.contains(MENTION, na=False)
    if os.path.exists(MANUAL_FILE):
        man = pd.read_csv(MANUAL_FILE)
        if "人工判定" in man.columns and man.人工判定.notna().any():
            key_cols = [c for c in ["clinician", "case"] if c in man.columns]
            sub_f = sub_f.merge(man[key_cols + ["人工判定"]], on=key_cols, how="left")
            sub_f["mentions"] = sub_f.人工判定.fillna(sub_f.auto).astype(bool)
            print(f"\n已读入人工编码 {MANUAL_FILE}（{int(man.人工判定.notna().sum())} 条），以人工判定为准")
        else:
            sub_f["mentions"] = sub_f.auto
    else:
        sub_f["mentions"] = sub_f.auto

    k, n = int(sub_f.mentions.sum()), len(sub_f)
    rate = k / n * 100
    ci = stats.binomtest(k, n).proportion_ci(0.95)
    print(f"\n在填写了依据的 {n} 条中，提及既往评估 {k} 条 = {rate:.1f}%"
          f"　95% CI [{ci.low*100:.1f}%, {ci.high*100:.1f}%]")
    print(f"AI 参照: {AI['disclosure_pct']:.2f}%"
          f"　比较 P = {stats.binomtest(k, n, AI['disclosure_pct']/100).pvalue:.2e}")
    if n < 60:
        print(f"\n  ⚠ 仅 {n} 条可用，置信区间很宽。这个比例只能作描述，")
        print("     不宜作为论文的主要证据；建议按上方补救方式之一加强。")
    g = sub_f.groupby("cond").mentions.agg(提及比例=lambda x: f"{x.mean()*100:.1f}%", n="size")
    if g.n.min() >= 15:
        print("\n按条件分列:"); print(g.to_string())
    else:
        print(f"\n按条件分列略（最小格仅 {int(g.n.min())} 条，不足以分列）")

    print("\n自动判定为『提及』的原话（请人工核对关键词是否误判）:")
    for t in sub_f[sub_f.mentions].reason_txt.head(8): print("  ·", t[:70])
    print("\n自动判定为『未提及』的原话（检查有无漏判）:")
    for t in sub_f[~sub_f.mentions].reason_txt.head(8): print("  ·", t[:70])
    out = sub_f[["clinician", "case", "cond", "reason_txt", "auto"]].copy()
    out["人工判定"] = ""
    out.to_csv("理由_待人工编码.csv", index=False, encoding="utf-8-sig")
    print(f"\n已写出 理由_待人工编码.csv（{len(out)} 条）。")
    print("人工在『人工判定』列填 1/0，另存为 理由_人工编码.csv 后重跑本格，即以人工结果为准。")

# ---- 不依赖自由文本的替代信号：有既往评估时，医生是否更倾向于写理由
r0 = clean(A[A.cond == CONDS[0]].reason).ne("").mean()
print(f"\n[替代信号] 填写依据的比例　无既往 {r0*100:.0f}%　vs 有既往 {sub.has_reason.mean()*100:.0f}%")
print("  （若有既往时明显更愿意写，说明既往评估确实进入了医生的推理过程）")

## 9. 分析六：把握度与易感性

In [ ]:
# ========== 分析六：把握度与易感性 ==========
ORDER = ["很不确定", "不太确定", "比较确定", "很确定"]
CONF_COL = "round1_conf"      # 想改用第 2 轮的把握度，把这里换成 "conf"

def conf_strata(df, col, label):
    """按把握度分层估计 β。df 用 B_FULL（不需要反事实基线）。"""
    if col not in df.columns:
        print(f"{label}：数据中没有 {col} 列，跳过。"); return
    v = df[col].dropna()
    if len(v) < 20:
        print(f"{label}：有效取值仅 {len(v)} 条，跳过。"); return
    lv = [x for x in ORDER if (v == x).sum() > 0]
    other = sorted(set(v.unique()) - set(ORDER))
    if other: print(f"{label}：发现非标准取值 {other}，已忽略。")
    print(f"{label}：{len(v)} 条，用到 {len(lv)} 个档位 → {dict(v.value_counts().reindex(lv).dropna().astype(int))}")
    if len(lv) < 2:
        print(f"  该评估者（或全体）在{label}上只用了『{lv[0] if lv else '—'}』一个档位，无法分层。")
        print("  这本身值得记录：把握度缺乏区分度，说明该量表在本任务中未被有效使用。")
        return
    d = df[df[col].isin(ORDER)].copy()
    d["conf1"] = pd.Categorical(d[col], ORDER, ordered=True)
    out, skipped = [], []
    for lab, sub_ in d.groupby("conf1", observed=True):
        if len(sub_) < 15:
            skipped.append(f"{lab}(n={len(sub_)})"); continue
        k, s_ = fe_beta(sub_)
        if np.isnan(k):
            skipped.append(f"{lab}(自由度不足)"); continue
        row = dict(把握度=str(lab), n=len(sub_), β=round(k, 3), CI=f"±{1.96*s_:.3f}")
        if "shift" in sub_.columns and np.isfinite(sub_["shift"]).any():
            row["平均绝对平移"] = round(sub_.loc[np.isfinite(sub_["shift"]), "shift"].abs().mean(), 2)
        out.append(row)
    print(pd.DataFrame(out).to_string(index=False) if out else "  各档均无法估计。")
    if skipped: print("  未能估计的档位:", "、".join(skipped))

# 用 B_FULL：β 的固定效应估计不需要反事实基线，
# 而 B 在医生人数少于 4 位时会被清空（每个有既往评估的病例都没有『无既往』观测）
src = B_FULL.copy()
if "shift" in B.columns and len(B):
    src = src.merge(B[["clinician", "case", "shift"]], on=["clinician", "case"], how="left")

print(f"可用观测 {len(src)} 条，来自 {src.clinician.nunique()} 位医生")
if src.clinician.nunique() < 4:
    print("⚠ 医生人数少于 4 位：病例固定效应会吃掉几乎全部自由度，本节基本无法估计。")
    print("  下面仍会尝试，并说明每一档失败的具体原因。\n")

conf_strata(src, CONF_COL, "第 1 轮把握度")
print()
conf_strata(src, "conf", "第 2 轮把握度")

print("\n预期：越不确定的判断越容易被既往评估推动（AI 上 βᵢ 在中等风险区间最高）")

## 10. 分析七：判别力对比（要主动报告）

In [ ]:
ev = A[A.dx36.isin(["CN", "MCI", "Dementia"])].copy()
ev["outcome"] = (ev.dx36 == "Dementia").astype(int)
print("各医生在『无既往』条件下的判别力:")
for c, gg in ev[ev.cond == CONDS[0]].groupby("clinician"):
    if gg.outcome.nunique() > 1 and len(gg) >= 8:
        print(f"  {c:12s} AUC {roc_auc_score(gg.outcome, gg.y):.3f}  (n={len(gg)})")
pooled = ev[ev.cond == CONDS[0]]
if pooled.outcome.nunique() > 1:
    print(f"\n合并（无既往条件）AUC = {roc_auc_score(pooled.outcome, pooled.y):.3f}  n={len(pooled)}")
print(f"AI 参照: 8 个模型 AUC 0.869–0.906")
print("\n注：医生每例只看到一个条件，样本量远小于 AI，AUC 比较只作描述，不作推断。")

## 11. 汇总图

In [ ]:
plt.rcParams.update({"font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"],
                     "font.size": 8, "axes.spines.top": False, "axes.spines.right": False,
                     "pdf.fonttype": 42, "axes.unicode_minus": False})
fig, ax = plt.subplots(1, 3, figsize=(10, 3.1))

# --- a: effect by attributed source
x = np.arange(3)
hv = [R1.loc[R1.条件 == c, "平移"].iloc[0] for c in CONDS[1:]]
he = [R1.loc[R1.条件 == c, "SE"].iloc[0] * 1.96 for c in CONDS[1:]]
av = [AI["self_"], AI["other_phys"], AI["other_ai"]]
ax[0].bar(x - 0.2, hv, 0.38, yerr=he, color="#4E8CC9", lw=0, capsize=2, label="Clinicians")
ax[0].bar(x + 0.2, av, 0.38, color="#C8553D", lw=0, label=f"AI ({AI['model']})")
ax[0].set_xticks(x); ax[0].set_xticklabels(["Self", "Physician", "AI system"])
ax[0].set_ylabel("Shift vs no prior (points)"); ax[0].axhline(0, color="#777", lw=0.6)
ax[0].legend(frameon=False); ax[0].set_title("a  Effect by attributed source", loc="left", fontweight="bold")

# --- b: key contrast with equivalence margin
ax[1].errorbar([est], [1], xerr=[[est - lo], [hi - est]], fmt="o", ms=6,
               color="#4E8CC9", capsize=3)
ax[1].errorbar([AI["self_minus_other"]], [0], xerr=1.96 * AI["self_minus_other_se"],
               fmt="s", ms=6, color="#C8553D", capsize=3)
ax[1].axvspan(-EQ_BOUND, EQ_BOUND, color="#EEEEEE", zorder=0)
ax[1].axvline(0, color="#777", lw=0.6, ls="--")
ax[1].set_yticks([0, 1]); ax[1].set_yticklabels(["AI", "Clinicians"]); ax[1].set_ylim(-0.6, 1.6)
ax[1].set_xlabel("Self \u2212 other (points)")
ax[1].set_title(f"b  Key contrast (grey = \u00b1{EQ_BOUND:g} equivalence margin)",
                loc="left", fontweight="bold")

# --- c: pull coefficient
k, se_k = fe_beta(B_FULL)
P = B[np.isfinite(B.gap) & np.isfinite(B["shift"])]
if len(P) >= 10 and not np.isnan(k):
    ax[2].scatter(P.gap, P["shift"], s=6, alpha=0.45, color="#4E8CC9", lw=0)
    b0 = P["shift"].mean() - k * P.gap.mean()      # 用均值反推截距，避免 polyfit 碰到缺失值
    xx = np.array([P.gap.min(), P.gap.max()])
    ax[2].plot(xx, b0 + k * xx, color="#222", lw=1)
else:
    ax[2].text(0.5, 0.5, "Insufficient baseline coverage\n(needs \u2265 4 clinicians)",
               ha="center", va="center", transform=ax[2].transAxes, fontsize=8, color="#777")
ax[2].axhline(0, color="#777", lw=0.6); ax[2].axvline(0, color="#777", lw=0.6)
ax[2].set_xlabel("Gap = prior \u2212 expected (points)"); ax[2].set_ylabel("Shift (points)")
ttl = "n/a" if np.isnan(k) else f"{k:.3f} \u00b1 {1.96*se_k:.3f}"
ax[2].set_title(f"c  Pull coefficient \u03b2 = {ttl}", loc="left", fontweight="bold")

plt.tight_layout(); plt.savefig("human_control_summary.pdf"); plt.show()
print("已保存 human_control_summary.pdf")

## 12. 结论速查

In [ ]:
print("="*72)
print(f"{'项目':<26}{'医生':>18}{'AI':>18}")
print("-"*72)
print(f"{'本人既往的拉力':<26}{R1.loc[0,'平移']:>+17.2f}{AI['self_']:>+18.2f}")
print(f"{'转诊医生的拉力':<26}{R1.loc[1,'平移']:>+17.2f}{AI['other_phys']:>+18.2f}")
print(f"{'AI 系统的拉力':<26}{R1.loc[2,'平移']:>+17.2f}{AI['other_ai']:>+18.2f}")
print(f"{'本人 − 他人':<26}{est:>+17.2f}{AI['self_minus_other']:>+18.2f}")
hb_ = R3.loc[0, "β"] if len(R3) else float("nan")
print(f"{'牵引系数 β':<26}{hb_:>17.3f}{AI['beta_median']:>18.3f}")
print(f"{'理由中提及既往':<26}{rate:>16.1f}%{AI['disclosure_pct']:>17.2f}%")
print("="*72)
if p_int < 0.05 and p_tost < 0.05:
    v = "医生的自我偏好可忽略，AI 显著更强 → 支持『AI 特有』"
elif p_int < 0.05:
    v = "医生也有自我偏好，但显著弱于 AI → 支持『AI 更强』，不支持『AI 特有』"
else:
    v = "医生与 AI 无显著差异 → 改以申报率与来源标签的性质差异立论"
print("\n主结论:", v)
print(f"申报率对比: 医生 {rate:.1f}% vs AI {AI['disclosure_pct']:.2f}% —— "
      f"{'差异明显，可独立支撑论点' if rate > 5 else '差异不明显，需谨慎表述'}")